# Explore the ribbon dataset

Look at the data before training anything. Most "the model is bad" problems are really data
problems: too few images in a class, one class photographed differently from the others,
mislabelled images, or the same ribbon in both train and test.

Prerequisite: `data/labels.csv` (from `ribbon manifest`) and ideally `data/splits.csv`
(from `ribbon split`). See docs/01_GETTING_STARTED.md, section 7.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from ribbon_classifier.config import load_config
from ribbon_classifier.dataset import load_image

config = load_config(os.environ.get("RIBBON_CONFIG"))  # default: configs/default.yaml
data_dir = config.path(config.paths.data_dir)
csv_path = next(
    (p for p in (config.path(config.paths.splits_csv), config.path(config.paths.labels_csv)) if p.exists()),
    None,
)
if csv_path is None:
    raise FileNotFoundError("No labels yet: build data/labels.csv with `ribbon manifest` first.")
df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
print(f"{csv_path.name}: {len(df)} images, {df['label'].nunique()} classes")
df.head()

## Class balance

Aim for at least ~50 images per class and no class more than ~5× larger than the smallest.
Heavily imbalanced data trains a model that is good at guessing the common class.

In [ ]:
counts = df["label"].value_counts()
counts.plot.barh(figsize=(8, max(3, 0.35 * len(counts))), title="Images per class")
plt.gca().invert_yaxis()
plt.show()
if "split" in df.columns:
    display(df.pivot_table(index="label", columns="split", values="image_path", aggfunc="count", fill_value=0))

## Image sizes

Tiny images (under ~200 px on the short side) carry little detail about folds and cuts.
Very different sizes between classes can become an unintended shortcut for the model.

In [ ]:
def size_of(relative_path):
    with Image.open(data_dir / relative_path) as image:
        return image.size


sizes = pd.DataFrame(df["image_path"].map(size_of).tolist(), columns=["width", "height"], index=df.index)
df = df.join(sizes)
display(df.groupby("label")[["width", "height"]].median().astype(int))
print("Images under 200 px on the short side:", int((df[["width", "height"]].min(axis=1) < 200).sum()))

## What each class looks like

Scan each row: are they all genuinely the same technique? Is anything besides the technique
(background, lighting, camera, zoom) consistently different between classes?

In [ ]:
PER_CLASS = 6
labels = sorted(df["label"].unique())
fig, axes = plt.subplots(len(labels), PER_CLASS, figsize=(2.2 * PER_CLASS, 2.4 * len(labels)), squeeze=False)
for row, label in zip(axes, labels, strict=True):
    sample = df[df["label"] == label].sample(min(PER_CLASS, (df["label"] == label).sum()), random_state=0)
    for ax in row:
        ax.axis("off")
    for ax, path in zip(row, sample["image_path"], strict=False):
        ax.imshow(load_image(data_dir / path))
    row[0].set_title(label, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Groups (leakage check)

A *group* is a set of images that must stay in one split, e.g. several photos of the same
physical ribbon. If every group has exactly one image, check whether that is really true:
near-duplicate photos in both train and test make test scores look better than reality.

In [ ]:
group_sizes = df.groupby("group").size()
print(f"{group_sizes.size} groups; {int((group_sizes == 1).sum())} with a single image; largest: {group_sizes.max()}")
if "split" in df.columns:
    crossing = df.groupby("group")["split"].nunique()
    print("Groups found in more than one split:", int((crossing > 1).sum()), "(must be 0)")